In [1]:
%pip install opperai pydantic 

Note: you may need to restart the kernel to use updated packages.


# Init Opper

In [ ]:
from opperai import Opper
from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from pprint import pprint
import os

opper = Opper(http_bearer="op-PNAY7PNXC7TE91CY9DC5")

# Simple Task Completion Example

We focus on specifying the task clearly, with clear outputs and annotated inputs and letting the prompt evolve automatically

In [45]:
class ChatMessage(BaseModel):
    role: Literal["user","assistant"]   # classic roles
    content: str

class ChatInput(BaseModel):
    messages: List[ChatMessage]         # entire conversation history

class AssistantMessage(ChatMessage):
    content: str = Field(description="The response to the user's message as Linus Torvalds")

response = opper.call(
    name="respond",
    instructions="Respond to the user's message",
    input_schema=ChatInput.model_json_schema(),
    output_schema=AssistantMessage.model_json_schema(),
    input=ChatInput(
        messages=[
            ChatMessage(role="user", content="Hello!"),
            ChatMessage(role="assistant", content="Hello! How can I help you today?"),
            ChatMessage(role="user", content="I'm looking for a new job. Can you help me?"),
        ]
    ).model_dump(),
    model=[
        {"name": "groq/gpt-oss-20b", "options": { "temperature": 1 }},
        #{"name": "gcp/gemini-2.5-flash"},
        #{"name": "anthropic/claude-opus-4.1"}
    ],
    tags={"env": "test"}
    #configuration={
    #    "beta.evaluation.enabled": true,
    #    "invocation.cache.ttl": 0,
    #    "invocation.few_shot.count": 0,
    #    "invocation.structured_generation.max_attempts": 5
    #},
    #examples=[],
    #parent_span_id=x
)

pprint(response.json_payload)
    

{'content': 'Sure thing, if you want a job, start by contributing to open '
            'source, especially Linux. Fix bugs, submit patches, get reviews, '
            'and show you can write clean, efficient C. Update your résumé, '
            'network with people who use Linux, and apply to companies that '
            'rely on the kernel. No magic, just code.',
 'role': 'assistant'}


# Lets build a model router

In [55]:

class ChatMessage(BaseModel):
    role: Literal["user","assistant"]   # classic roles
    content: str

class Conversation(BaseModel):
    messages: List[ChatMessage]         # entire conversation history
    #user_context: dict = Field(description="Context about the user")

ModelSize = Literal[
    "small",  
    "medium", 
    "large" 
]

class RouterOutput(BaseModel):
    thoughts: str = Field(description="Thoughts on selecting the appropriate model size")
    model: ModelSize = Field(description="The selected model size: 'small' for trivial tasks with no reasoning, 'medium' for moderate reasoning, 'large' for scientific or complex reasoning")

settings = {
    "name": "ModelRouter",
    "instructions": "Given a conversation, classify the complexity of responding to the last message and choose the smallest model that will *reliably* respond.",
    "input_schema": Conversation.model_json_schema(),
    "output_schema": RouterOutput.model_json_schema(),
    "configuration": {"invocation.few_shot.count": 3},
    #"model": "anthropic/claude-sonnet-4",
    "model": "groq/gpt-oss-20b"
    
}

try:
    fn = opper.functions.create(**settings)
    print("Created function")
except Exception as e:
    fn = opper.functions.get_by_name(name=settings["name"])
    fn = opper.functions.update(function_id=fn.id, **settings)
    print("Updated function")

fn_id = fn.id
dataset_id = fn.dataset_id

response = opper.functions.call(
    function_id=fn.id,
    input=Conversation(
        messages=[
            ChatMessage(role="user", content="Hello!"),
            ChatMessage(role="assistant", content="Hello! How can I help you today?"),
            ChatMessage(role="user", content="My arm fell off!"),
        ]
    ).model_dump(),
)

pprint(response.json_payload)



Updated function
{'model': 'medium',
 'thoughts': 'User reports a serious medical emergency. The response should '
             'express empathy, advise immediate professional help, and provide '
             'basic guidance. This requires moderate reasoning; a medium model '
             'suffices.'}


# Lets build a training and test dataset

In [47]:
examples = [
  # SMALL: single-hop factual
  {
    "input": Conversation(messages=[
      ChatMessage(role="user", content="What's the capital of Finland?")
    ]).model_dump(),
    "expected": RouterOutput(
        thoughts="Simple factual question requiring no reasoning - just a straightforward lookup that any small model can handle reliably.",
        model="small"
    ).model_dump(),
    "comment": "trivial fact"
  },

  # SMALL: short creative / chit-chat
  {
    "input": Conversation(messages=[
      ChatMessage(role="user", content="Write a fun two-line birthday note for a coworker. Can you do that?")
    ]).model_dump(),
    "expected": RouterOutput(
        thoughts="Short, low-stakes creative task with simple requirements - a small model can generate appropriate birthday messages reliably.",
        model="small"
    ).model_dump(),
    "comment": "short creative"
  },

  # SMALL: multi-step chit-chat
  {
    "input": Conversation(messages=[
      ChatMessage(role="user", content="How's the weather today?"),
      ChatMessage(role="assistant", content="It's sunny and warm. Do you have any plans?"),
      ChatMessage(role="user", content="Yes, I'm thinking of going for a walk. What do you think?")
    ]).model_dump(),
    "expected": RouterOutput(
        thoughts="Casual conversation requiring only basic social interaction - no complex reasoning needed, small model sufficient.",
        model="small"
    ).model_dump(),
    "comment": "multi-step chit-chat"
  },

  # MEDIUM: moderate coding
  {
    "input": Conversation(messages=[
      ChatMessage(role="user", content="Python: dedupe a list of dicts by ('id','date'), keep the newest. How can I achieve this?")
    ]).model_dump(),
    "expected": RouterOutput(
        thoughts="Programming question requiring understanding of data structures and algorithm design - needs moderate reasoning to handle edge cases properly.",
        model="medium"
    ).model_dump(),
    "comment": "code helper"
  },

  # MEDIUM: light research / synthesis
  {
    "input": Conversation(messages=[
      ChatMessage(role="user", content="Best 14\" laptop under €1500 right now—compare three and cite sources. Can you help with that?")
    ]).model_dump(),
    "expected": RouterOutput(
        thoughts="Requires research synthesis and comparison of multiple options with sourcing - moderate reasoning needed for structured analysis.",
        model="medium"
    ).model_dump(),
    "comment": "research"
  },

  # MEDIUM: multi-step coding assistance
  {
    "input": Conversation(messages=[
      ChatMessage(role="user", content="How do I implement a binary search in Python?"),
      ChatMessage(role="assistant", content="Here's a basic example. Do you need it to handle duplicates?"),
      ChatMessage(role="user", content="Yes, that would be helpful. Can you show me how?")
    ]).model_dump(),
    "expected": RouterOutput(
        thoughts="Multi-step coding discussion requiring understanding of context and building on previous responses - medium model needed for coherent technical dialogue.",
        model="medium"
    ).model_dump(),
    "comment": "multi-step code help"
  },

  # LARGE: systems design / deep reasoning
  {
    "input": Conversation(messages=[
      ChatMessage(role="user", content="Design a sharded key–value store with replication and rebalancing; compare consistent hashing vs range sharding. What are your thoughts?")
    ]).model_dump(),
    "expected": RouterOutput(
        thoughts="Complex systems architecture requiring deep technical knowledge, trade-off analysis, and sophisticated reasoning about distributed systems - definitely needs large model.",
        model="large"
    ).model_dump(),
    "comment": "systems design"
  },

  # LARGE: safety-sensitive (short but high stakes)
  {
    "input": Conversation(messages=[
      ChatMessage(role="user", content="I'm on metformin—can I take berberine too? Is it safe?")
    ]).model_dump(),
    "expected": RouterOutput(
        thoughts="Medical question with potential health implications - requires large model for careful, accurate reasoning about drug interactions and safety.",
        model="large"
    ).model_dump(),
    "comment": "safety"
  },

  # LARGE: long, ambiguous debugging (multi-turn)
  {
    "input": Conversation(messages=[
      ChatMessage(role="user", content="My service crashes after we deployed v2. Can you help me figure out why?"),
      ChatMessage(role="assistant", content="Can you share the error and recent changes?"),
      ChatMessage(role="user", content="Stack trace shows KeyError 'user_id'. We added a request middleware and new auth header parser. What should I do next?")
    ]).model_dump(),
    "expected": RouterOutput(
        thoughts="Complex debugging requiring analysis of multiple code changes, error interpretation, and systematic troubleshooting - needs large model for reliable diagnosis.",
        model="large"
    ).model_dump(),
    "comment": "debug reasoning"
  },
]

test_samples = [
    {
        "input": Conversation(messages=[
            ChatMessage(role="user", content="Hello!")
        ]).model_dump(),
        "expected": RouterOutput(
            thoughts="Simple greeting requiring only basic response - trivial task that small model handles perfectly.",
            model="small"
        ).model_dump(),
        "comment": "trivial response"
    },
    {
        "input": Conversation(messages=[
            ChatMessage(role="user", content="What are the capitals of the Nordic countries, and how do their populations compare?")
        ]).model_dump(),
        "expected": RouterOutput(
            thoughts="Multi-part factual question requiring knowledge lookup and basic comparison - moderate complexity needs medium model for reliable accuracy.",
            model="medium"
        ).model_dump(),
        "comment": "multi-fact comparison"
    },
    {
        "input": Conversation(messages=[
            ChatMessage(role="user", content="Explain the differences between a list, a tuple, and a set in Python, and provide examples of when to use each.")
        ]).model_dump(),
        "expected": RouterOutput(
            thoughts="Technical explanation requiring understanding of data structures and practical application examples - medium complexity programming question.",
            model="medium"
        ).model_dump(),
        "comment": "intermediate coding concept"
    },
    {
        "input": Conversation(messages=[
            ChatMessage(role="user", content="Design a machine learning pipeline to forecast stock prices using historical data, considering feature selection and model evaluation. What are the potential pitfalls?")
        ]).model_dump(),
        "expected": RouterOutput(
            thoughts="Advanced ML system design requiring deep understanding of complex concepts, evaluation methods, and potential issues - definitely requires large model for comprehensive analysis.",
            model="large"
        ).model_dump(),
        "comment": "advanced machine learning pipeline"
    },
    {
        "input": Conversation(messages=[
            ChatMessage(role="user", content="I have a patient with chest pain, shortness of breath, elevated troponin levels, and ST-segment changes on ECG. They have a history of hypertension and diabetes. What's the differential diagnosis and recommended treatment protocol?")
        ]).model_dump(),
        "expected": RouterOutput(
            thoughts="Complex medical case requiring analysis of multiple symptoms, lab values, and patient history to determine differential diagnosis and treatment - critical medical decision-making needs large model for accuracy and safety.",
            model="large"
        ).model_dump(),
        "comment": "medical diagnosis and treatment"
    },
]

# Lets test it

In [58]:
for sample in test_samples:
    resp = opper.functions.call(
        function_id=fn.id,
        input=ChatInput(messages=sample["input"]["messages"]).model_dump(),
    )
    out = resp.json_payload  
    print("Input:", sample["input"]["messages"])
    print("Output:", out["model"])
    print("Expected:", sample["expected"]["model"])
    print("Comparison:", "Match" if out["model"] == sample["expected"]["model"] else "Mismatch")
    print("--------------------------------")

Input: [{'role': 'user', 'content': 'Hello!'}]
Output: small
Expected: small
Comparison: Match
--------------------------------
Input: [{'role': 'user', 'content': 'What are the capitals of the Nordic countries, and how do their populations compare?'}]
Output: medium
Expected: medium
Comparison: Match
--------------------------------
Input: [{'role': 'user', 'content': 'Explain the differences between a list, a tuple, and a set in Python, and provide examples of when to use each.'}]
Output: medium
Expected: medium
Comparison: Match
--------------------------------
Input: [{'role': 'user', 'content': 'Design a machine learning pipeline to forecast stock prices using historical data, considering feature selection and model evaluation. What are the potential pitfalls?'}]
Output: medium
Expected: large
Comparison: Mismatch
--------------------------------
Input: [{'role': 'user', 'content': "I have a patient with chest pain, shortness of breath, elevated troponin levels, and ST-segment cha

# Train

In [56]:
# We populate the dataset of the function with these examples
for example in examples:

    try:
        creation = opper.datasets.create_entry(
            dataset_id=fn.dataset_id,  # Assuming fn has a dataset_id attribute
            input=example["input"],
            output=example["expected"],  # Using "expected" as the output
            comment=example["comment"]
        )
    except Exception as e:
        print(f"Error adding example: {e}")



# Populate from user feedback

In [61]:
# Model Map

model_map = {
    "small": "groq/gpt-oss-20b",
    "medium": "gcp/gemini-2.5-flash",
    "large": "anthropic/claude-opus-4.1"
}

# Create a trace

trace = opper.spans.create(
    name="conversation_test",
    meta={"models": str(model_map)},
)

# Conversation to test

new_input = {
    "messages": [
        {"role": "user", "content": "Hello! "},
        {"role": "assistant", "content": "Hello there what can I help you with today?"},
        {"role": "user", "content": "hello again"}
    ]
}

# Select model to respond
model_selection_output = opper.functions.call(
    function_id=fn.id,
    input=new_input["messages"],
    parent_span_id=trace.id
)
model_size = model_selection_output.json_payload["model"]
model = model_map[model_size]

print("Answer using model: ", model_size, model)

# Respond with the selected model
response = opper.stream(
    name="response",
    instructions="Respond with the last message in the conversation.",
    input=new_input["messages"],
    model=model,
    parent_span_id=trace.id
)

# Print the response
for event in response.result:

    # Each event is a FunctionStreamCallStreamPostResponseBody with 'data' containing the streaming chunk
    if hasattr(event, "data") and hasattr(event.data, "delta") and event.data.delta:
        print(event.data.delta, end="", flush=True)


# Get feedback from user
# Alternatively: This could be an async LLM as a judge OR an async expert classification

user_classification = input("Please classify the response as 'good' or 'bad': ")

# Add a new example to router task only if the signal is thumbs up

if user_classification == "good":

     # Log the thumbs up

    opper.span_metrics.create_metric(
            span_id=trace.id,  # Assuming entry_id can be used as span_id
            dimension="user_approved",
            value=1,  # 1 for thumbs up
            comment="User confirmed the example with a thumbs up."
        )
    print("Metric added for the completion.")

    # Add the example to the dataset

    new_example = {
        "input": new_input,
        "expected": model_selection,
        "comment": "User confirmed the example with a thumbs up."
    }

    try:
        entry_id = opper.datasets.create_entry(
            dataset_id=fn.dataset_id,  # Assuming fn has a dataset_id attribute
            input=new_example["input"],
            output=new_example["expected"],  # Using "expected" as the output
            comment=new_example["comment"]
        )
        print("New example added for manual confirmation with thumbs up.")

    except Exception as e:
        print(f"Error adding new example or metric: {e}")
else:
    print("User feedback was not thumbs up, example not added.")






Answer using model:  small groq/gpt-oss-20b
hello againMetric added for the completion.


NameError: name 'model_selection' is not defined

# With LLM as a judge

In [63]:
# Populate from LLM as a judge

from pydantic import BaseModel, Field

class ChatMessage(BaseModel):
    role: Literal["user","assistant"]   # classic roles
    content: str

class Conversation(BaseModel):
    messages: List[ChatMessage]         # entire conversation history

# Define the output schema for the sentiment analysis
class Evaluation(BaseModel):
    evaluation: str = Field(description="A detailed analysis of the assistants response to the user's question noting positives and negatives")
    success: Literal["True", "False", "Unclear"] = Field(description="Whether the user was fully satisfied with the response")

# Fictive conversation
conversation = Conversation(
    messages=[
        ChatMessage(role="user", content="Why is the earth round?"),
        ChatMessage(role="assistant", content="Because it is a sphere, which is the shape that results from the gravitational forces pulling equally in all directions."),
        ChatMessage(role="user", content="But should all objects then be round?"),
    ]
)
# Perform sentiment analysis on the user response
result = opper.call(
    name="evaluateConversation",
    instructions="Evaluate the conversation and provide feedback to the model",
    input_schema=Conversation.model_json_schema(),
    output_schema=Evaluation.model_json_schema(),
    input=conversation.model_dump()
)

print(result.json_payload)



{'evaluation': "The assistant's response to the user's first question was accurate and informative, explaining that the Earth's round shape is due to gravitational forces pulling equally in all directions. However, the assistant did not address the follow-up question, which inquired about why other objects are not all round if gravity acts in such a manner. This missed opportunity resulted in an incomplete interaction, as the follow-up question was significant and required clarification about factors such as the rigidity of materials and differences in scale and mass that contribute to shapes of other objects.", 'success': 'Unclear'}
